# Chapter 7 Companion Notebook: Time Series Analysis and Forecasting

This notebook reproduces every worked numerical example from Chapter 7 of *AI in Finance*: the correlogram, the Augmented Dickey-Fuller test, the AR(1) fit and forecast (with confidence intervals and p-values), multi-step forecasting, the Ljung-Box test and an AR(1)-versus-AR(2) AIC/BIC comparison, exponential smoothing, cointegration and pairs trading, vector autoregression (impulse response and eigenvalue stability), the GARCH(1,1) variance recursion (reusing Chapter 2's AssetA returns), walk-forward validation, and an RMSE/MAPE comparison against a naive benchmark.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. The interest-rate deviation series and its correlogram

In [1]:
import numpy as np
import pandas as pd

X = np.array([9.0, 8.0, 7.0, 7.0, 5.0, 6.0, 4.0, 4.0])
mean_all = X.mean()
denom = np.sum((X - mean_all) ** 2)

def acf(k, x=X, mean=mean_all, denom=denom):
    return np.sum((x[k:] - mean) * (x[:-k] - mean)) / denom

for k in [1, 2, 3]:
    print(f"ACF({k}) = {acf(k):.4f}")

ACF(1) = 0.4973
ACF(2) = 0.2394
ACF(3) = 0.0346


The roughly geometric decay (about halving at each lag) is the correlogram signature of an AR(1) process, matching Figure 7.1 in the chapter.

## 2. Testing for stationarity: the Augmented Dickey-Fuller test (Section 7.4)

Regress $\Delta X_t$ on $X_{t-1}$ and test $H_0: \gamma=0$ (a unit root) against the nonstandard Dickey-Fuller critical value.

In [2]:
dX = X[1:] - X[:-1]
Xlag_adf = X[:-1]

xbar_adf = Xlag_adf.mean()
gamma_hat = np.sum((Xlag_adf - xbar_adf) * (dX - dX.mean())) / np.sum((Xlag_adf - xbar_adf) ** 2)
alpha_hat = dX.mean() - gamma_hat * xbar_adf

fitted_adf = alpha_hat + gamma_hat * Xlag_adf
resid_adf = dX - fitted_adf
dof_adf = len(dX) - 2
se_gamma = np.sqrt(np.sum(resid_adf ** 2) / dof_adf / np.sum((Xlag_adf - xbar_adf) ** 2))
t_stat = gamma_hat / se_gamma

print(f"gamma_hat = {gamma_hat:.4f}")
print(f"SE(gamma_hat) = {se_gamma:.4f}")
print(f"t-statistic = {t_stat:.4f}")

df_critical_5pct = -2.89
print(f"5% Dickey-Fuller critical value: {df_critical_5pct}")
print(f"Reject unit root? {'Yes' if t_stat < df_critical_5pct else 'No'}")

gamma_hat = -0.2903
SE(gamma_hat) = 0.2589
t-statistic = -1.1215
5% Dickey-Fuller critical value: -2.89
Reject unit root? No


## 3. Fitting the AR(1) model and forecasting (Section 7.6)

In [3]:
Xt = X[1:]
Xlag = X[:-1]
xbar, ybar = Xlag.mean(), Xt.mean()

phi = np.sum((Xlag - xbar) * (Xt - ybar)) / np.sum((Xlag - xbar) ** 2)
c = ybar - phi * xbar

fitted = c + phi * Xlag
resid = Xt - fitted
sse = np.sum(resid ** 2)
sst = np.sum((Xt - ybar) ** 2)
r2 = 1 - sse / sst

print(f"phi = {phi:.4f}, c = {c:.4f}, R^2 = {r2:.4f}")
print(f"Implied long-run mean c/(1-phi) = {c/(1-phi):.3f}")
print(f"Sample mean of full series: {mean_all:.2f}")

X9_forecast = c + phi * X[-1]
print(f"One-step-ahead forecast X9: {X9_forecast:.3f}")

phi = 0.7097, c = 1.1935, R^2 = 0.6005
Implied long-run mean c/(1-phi) = 4.111
Sample mean of full series: 6.25
One-step-ahead forecast X9: 4.032


### Confidence intervals and p-values for $\hat\phi$ (Section 7.6.1)

Also verify $\hat\gamma = \hat\phi - 1$, connecting back to the ADF test above.

In [4]:
from scipy import stats

dof_ar1 = len(Xt) - 2
sigma2_ar1 = sse / dof_ar1
se_phi = np.sqrt(sigma2_ar1 / np.sum((Xlag - xbar) ** 2))
se_c = np.sqrt(sigma2_ar1 * (1 / len(Xt) + xbar ** 2 / np.sum((Xlag - xbar) ** 2)))

t_phi = phi / se_phi
t_c = c / se_c
p_phi = 2 * (1 - stats.t.cdf(abs(t_phi), dof_ar1))
p_c = 2 * (1 - stats.t.cdf(abs(t_c), dof_ar1))
tcrit_ar1 = stats.t.ppf(0.975, dof_ar1)
ci_phi = (phi - tcrit_ar1 * se_phi, phi + tcrit_ar1 * se_phi)

print(f"SE(phi)={se_phi:.4f}, t={t_phi:.2f}, p={p_phi:.4f}")
print(f"95% CI for phi: ({ci_phi[0]:.2f}, {ci_phi[1]:.2f})")
print(f"SE(c)={se_c:.4f}, t={t_c:.2f}, p={p_c:.4f}")
print(f"\nCheck: gamma_hat={gamma_hat:.4f}, phi-1={phi-1:.4f}")

SE(phi)=0.2589, t=2.74, p=0.0407
95% CI for phi: (0.04, 1.38)
SE(c)=1.7503, t=0.68, p=0.5256

Check: gamma_hat=-0.2903, phi-1=-0.2903


## 4. Multi-step forecasting and the growth of uncertainty (Section 7.7)

Recursively forecast horizons $h=1,2,3,5,10$ from $X_8=4.0$, and check convergence toward the long-run mean.

In [5]:
horizons = [1, 2, 3, 5, 10]
forecasts = {}
x_current = X[-1]
h_forecast = 0
x_h = x_current
for h in range(1, max(horizons) + 1):
    x_h = c + phi * x_h
    if h in horizons:
        forecasts[h] = x_h

for h, val in forecasts.items():
    print(f"h={h:2d} -> X_hat_(8+h) = {val:.2f}")

long_run_mean = c / (1 - phi)
print(f"\nLong-run mean c/(1-phi) = {long_run_mean:.3f}")

variance_ratio_limit = 1 / (1 - phi ** 2)
print(f"Limiting variance ratio 1/(1-phi^2) = {variance_ratio_limit:.2f}")

h= 1 -> X_hat_(8+h) = 4.03
h= 2 -> X_hat_(8+h) = 4.06
h= 3 -> X_hat_(8+h) = 4.07
h= 5 -> X_hat_(8+h) = 4.09
h=10 -> X_hat_(8+h) = 4.11

Long-run mean c/(1-phi) = 4.111
Limiting variance ratio 1/(1-phi^2) = 2.01


## 4b. Diagnosing the fit: the Ljung-Box test and AR(1) vs. AR(2) (Section 7.7.1)

Check whether the AR(1) residuals are white noise, then compare AR(1) to AR(2) via AIC/BIC on a common sample.

In [6]:
# Ljung-Box test on the AR(1) residuals computed above
n_lb = len(resid)
rbar_lb = resid.mean()
ss_lb = np.sum((resid - rbar_lb) ** 2)

def resid_acf(k):
    return np.sum((resid[k:] - rbar_lb) * (resid[:-k] - rbar_lb)) / ss_lb

rho1_lb, rho2_lb = resid_acf(1), resid_acf(2)
print(f"Residuals: {resid.round(4)}")
print(f"Residual ACF: rho1={rho1_lb:.4f}, rho2={rho2_lb:.4f}")

h_lb = 2
Q_lb = n_lb * (n_lb + 2) * sum(resid_acf(k) ** 2 / (n_lb - k) for k in range(1, h_lb + 1))
dof_lb = h_lb - 1  # subtract p=1 AR parameter
p_lb = 1 - stats.chi2.cdf(Q_lb, df=dof_lb)
print(f"Ljung-Box Q({h_lb}) = {Q_lb:.4f}, dof={dof_lb}, p-value = {p_lb:.4f}")

# AR(1) vs AR(2) on a common six-observation sample (t=3..8)
y_common = X[2:]
x1_common = X[1:-1]
x2_common = X[:-2]
n_common = len(y_common)

X1_common = np.column_stack([np.ones(n_common), x1_common])
b1_common, *_ = np.linalg.lstsq(X1_common, y_common, rcond=None)
ssr1_common = np.sum((y_common - X1_common @ b1_common) ** 2)

X2_common = np.column_stack([np.ones(n_common), x1_common, x2_common])
b2_common, *_ = np.linalg.lstsq(X2_common, y_common, rcond=None)
ssr2_common = np.sum((y_common - X2_common @ b2_common) ** 2)

for label, ssr, k in [("AR(1)", ssr1_common, 2), ("AR(2)", ssr2_common, 3)]:
    aic = n_common * np.log(ssr / n_common) + 2 * k
    bic = n_common * np.log(ssr / n_common) + k * np.log(n_common)
    print(f"{label}: SSR={ssr:.4f}, AIC={aic:.2f}, BIC={bic:.2f}")

print(f"\nAR(2) coefficients: c={b2_common[0]:.4f}, phi1={b2_common[1]:.4f}, phi2={b2_common[2]:.4f}")

Residuals: [ 0.4194  0.129   0.8387 -1.1613  1.2581 -1.4516 -0.0323]
Residual ACF: rho1=-0.6827, rho2=0.4890
Ljung-Box Q(2) = 7.9059, dof=1, p-value = 0.0049
AR(1): SSR=5.6000, AIC=3.59, BIC=3.17
AR(2): SSR=1.3933, AIC=-2.76, BIC=-3.39

AR(2) coefficients: c=-0.8427, phi1=0.0337, phi2=0.8764


## 5. Exponential smoothing (Section 7.8)

In [7]:
alpha = 0.4
smoothed = [X[0]]
for t in range(1, len(X)):
    smoothed.append(alpha * X[t] + (1 - alpha) * smoothed[-1])

next_forecast = alpha * X[-1] + (1 - alpha) * smoothed[-1]
smoothed.append(next_forecast)

comparison = pd.DataFrame({
    'Month': list(range(1, 10)),
    'Observed': list(X) + [np.nan],
    'Exp. smoothed forecast': np.round(smoothed, 3),
})
comparison

,Month,Observed,Exp. smoothed forecast
0,1,9.0,9.000
1,2,8.0,8.600
2,3,7.0,7.960
3,4,7.0,7.576
4,5,5.0,6.546
5,6,6.0,6.327
6,7,4.0,5.396
7,8,4.0,4.838
8,9,NaN,4.503


Compare the exponential-smoothing forecast for month 9 to the AR(1) forecast computed above: the two methods use the data differently and need not agree.

## 6. Cointegration and pairs trading (Section 7.10)

Two hypothetical bank stocks. Test whether the spread is stationary via the Engle-Granger two-step method, then compute a $z$-score for the trading rule.

In [8]:
stock_a = np.array([50, 51, 53, 52, 54, 55], dtype=float)
stock_b = np.array([48, 49, 50, 50, 51, 52], dtype=float)
spread = stock_a - stock_b

print(f"Spread: {spread}")
print(f"Mean spread: {spread.mean():.2f}, Std spread: {spread.std(ddof=1):.3f}")

# Engle-Granger step 1: regress A on B
b_bar, a_bar = stock_b.mean(), stock_a.mean()
beta_hat = np.sum((stock_b - b_bar) * (stock_a - a_bar)) / np.sum((stock_b - b_bar) ** 2)
alpha_hat_eg = a_bar - beta_hat * b_bar
print(f"\nCointegrating regression: alpha_hat={alpha_hat_eg:.2f}, beta_hat={beta_hat:.2f}")

# z-score of the final spread
z_final = (spread[-1] - spread.mean()) / spread.std(ddof=1)
print(f"\nFinal spread z-score: {z_final:.2f}")

Spread: [2. 2. 3. 2. 3. 3.]
Mean spread: 2.50, Std spread: 0.548

Cointegrating regression: alpha_hat=-12.50, beta_hat=1.30

Final spread z-score: 0.91


## 6b. Vector autoregression: impulse response and stability (Section 7.11)

A fitted VAR(1) with coefficients $a_{11}=0.6$, $a_{12}=0.2$, $a_{21}=0.1$, $a_{22}=0.7$. Trace a one-unit shock to $X$ (interest rate) through both series, and check the system's stability via the eigenvalues of the coefficient matrix.

In [9]:
A_var = np.array([[0.6, 0.2], [0.1, 0.7]])

Xv, Yv = 1.0, 0.0  # unit shock to X, Y unshocked
irf = [(0, Xv, Yv)]
for t in range(1, 5):
    Xv, Yv = A_var @ [Xv, Yv]
    irf.append((t, Xv, Yv))

irf_df = pd.DataFrame(irf, columns=['t', 'X_response', 'Y_response'])
print(irf_df.round(4).to_string(index=False))

eigenvalues = np.linalg.eigvals(A_var)
print(f"\nEigenvalues of A: {eigenvalues}")
print(f"Stable (all |eigenvalue| < 1)? {np.all(np.abs(eigenvalues) < 1)}")

 t  X_response  Y_response
 0      1.0000      0.0000
 1      0.6000      0.1000
 2      0.3800      0.1300
 3      0.2540      0.1290
 4      0.1782      0.1157

Eigenvalues of A: [0.5 0.8]
Stable (all |eigenvalue| < 1)? True


## 6c. PCA and forecasting the term structure of interest rates (Section 7.12)

Real U.S. Treasury daily par yield curve rates, 1990-2023 (`ch7_data/yield-curve-rates-1990-2023.csv`).

In [10]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm

yc_df = pd.read_csv("ch7_data/yield-curve-rates-1990-2023.csv")
yc_df["Date"] = pd.to_datetime(yc_df["Date"], format="mixed")
yc_df = yc_df.sort_values("Date").reset_index(drop=True)

yc_cols = ["3 Mo", "6 Mo", "1 Yr", "2 Yr", "3 Yr", "5 Yr", "7 Yr", "10 Yr"]
yc_sub = yc_df[["Date"] + yc_cols].dropna().reset_index(drop=True)
print(f"Clean panel: {yc_sub.shape[0]} daily observations, {len(yc_cols)} maturities, "
      f"{yc_sub.Date.min().date()} to {yc_sub.Date.max().date()}")

yc_X = yc_sub[yc_cols].values
yc_Xc = yc_X - yc_X.mean(axis=0)

yc_cov = np.cov(yc_Xc, rowvar=False)
yc_eigvals, yc_eigvecs = np.linalg.eigh(yc_cov)
yc_order = np.argsort(yc_eigvals)[::-1]
yc_eigvals, yc_eigvecs = yc_eigvals[yc_order], yc_eigvecs[:, yc_order]

yc_explained_var_ratio = yc_eigvals / yc_eigvals.sum()
print("\nExplained variance ratio, PC1-PC5:")
for i in range(5):
    print(f"  PC{i+1}: {yc_explained_var_ratio[i]:.4%}")
print(f"  Cumulative (PC1-3): {yc_explained_var_ratio[:3].sum():.4%}")

if yc_eigvecs[:, 0].mean() < 0:
    yc_eigvecs[:, 0] *= -1
yc_loadings = pd.DataFrame(yc_eigvecs[:, :3], index=yc_cols, columns=["PC1", "PC2", "PC3"])
print("\nLoadings (PC1-PC3) by maturity:")
print(yc_loadings.round(4))

yc_scores = pd.DataFrame(yc_Xc @ yc_eigvecs[:, :3], columns=["PC1", "PC2", "PC3"], index=yc_sub["Date"])

yc_adf_level = adfuller(yc_scores["PC1"], autolag="AIC")
print(f"\nADF on PC1 level: stat={yc_adf_level[0]:.4f}, p-value={yc_adf_level[1]:.4f}")

yc_pc1_diff = yc_scores["PC1"].diff().dropna()
yc_adf_diff = adfuller(yc_pc1_diff, autolag="AIC")
print(f"ADF on differenced PC1: stat={yc_adf_diff[0]:.4f}, p-value={yc_adf_diff[1]:.4f}")

yc_y, yc_x = yc_pc1_diff.values[1:], yc_pc1_diff.values[:-1]
yc_model = sm.OLS(yc_y, sm.add_constant(yc_x)).fit()
print(f"\nAR(1) on differenced PC1: const={yc_model.params[0]:.6f}, phi={yc_model.params[1]:.6f}, "
      f"p-value={yc_model.pvalues[1]:.4f}, R^2={yc_model.rsquared:.4f}")

yc_last_diff = yc_pc1_diff.values[-1]
yc_next_diff_forecast = yc_model.params[0] + yc_model.params[1] * yc_last_diff
yc_next_level_forecast = yc_scores["PC1"].values[-1] + yc_next_diff_forecast
print(f"\nLast PC1 level ({yc_sub.Date.iloc[-1].date()}): {yc_scores['PC1'].values[-1]:.4f}")
print(f"Last daily change in PC1: {yc_last_diff:.4f}")
print(f"Forecast next-day change in PC1: {yc_next_diff_forecast:.4f}")
print(f"Forecast next-day PC1 level: {yc_next_level_forecast:.4f}")

yc_implied_yield_change = yc_eigvecs[:, 0] * yc_next_diff_forecast
print("\nImplied one-day yield change by maturity, from the PC1 forecast alone:")
for c, v in zip(yc_cols, yc_implied_yield_change):
    print(f"  {c}: {v:+.4f} pp")

Clean panel: 8503 daily observations, 8 maturities, 1990-01-02 to 2023-12-29

Explained variance ratio, PC1-PC5:
  PC1: 95.9534%
  PC2: 3.7813%
  PC3: 0.2223%
  PC4: 0.0281%
  PC5: 0.0073%
  Cumulative (PC1-3): 99.9570%

Loadings (PC1-PC3) by maturity:
          PC1     PC2     PC3
3 Mo   0.3618 -0.4161  0.5210
6 Mo   0.3686 -0.3940  0.2112
1 Yr   0.3706 -0.3025 -0.1491
2 Yr   0.3757 -0.0815 -0.4574
3 Yr   0.3675  0.0735 -0.4569
5 Yr   0.3459  0.3105 -0.1504
7 Yr   0.3274  0.4315  0.1315
10 Yr  0.3047  0.5342  0.4515



ADF on PC1 level: stat=-2.2950, p-value=0.1736


ADF on differenced PC1: stat=-16.4736, p-value=0.0000

AR(1) on differenced PC1: const=-0.001117, phi=0.041064, p-value=0.0002, R^2=0.0017

Last PC1 level (2023-12-29): 3.0632
Last daily change in PC1: -0.0228
Forecast next-day change in PC1: -0.0021
Forecast next-day PC1 level: 3.0611

Implied one-day yield change by maturity, from the PC1 forecast alone:
  3 Mo: -0.0007 pp
  6 Mo: -0.0008 pp
  1 Yr: -0.0008 pp
  2 Yr: -0.0008 pp
  3 Yr: -0.0008 pp
  5 Yr: -0.0007 pp
  7 Yr: -0.0007 pp
  10 Yr: -0.0006 pp


## 7. GARCH(1,1) variance recursion using Chapter 2's AssetA returns (Section 7.12)

In [11]:
r = np.array([0.0200, -0.0098, 0.0297, -0.0096])
omega, alpha_g, beta_g = 0.00002, 0.10, 0.85

sigma2 = np.zeros(5)
sigma2[0] = 0.000414  # seed with the sample variance from Chapter 2
for t in range(1, 5):
    sigma2[t] = omega + alpha_g * r[t - 1] ** 2 + beta_g * sigma2[t - 1]

garch_table = pd.DataFrame({
    't': range(1, 6),
    'sigma^2': sigma2.round(6),
    'sigma (vol)': np.sqrt(sigma2).round(4),
})
print(garch_table.to_string(index=False))

long_run_var = omega / (1 - alpha_g - beta_g)
print(f"\nLong-run variance: {long_run_var:.4f}, long-run volatility: {np.sqrt(long_run_var):.2%}")

 t  sigma^2  sigma (vol)
 1 0.000414       0.0203
 2 0.000412       0.0203
 3 0.000380       0.0195
 4 0.000431       0.0208
 5 0.000396       0.0199

Long-run variance: 0.0004, long-run volatility: 2.00%


Volatility rises from 1.95% to 2.08% immediately after the large return of 2.97% at $t=3$, then decays back toward the long-run level, exactly the volatility-clustering pattern GARCH is designed to capture.

## 8. Walk-forward validation, illustrated (Section 7.14)

Rather than a single train/test split, walk-forward validation refits the model at each step using only data available up to that point, and scores a genuine one-step-ahead forecast each time.

In [12]:
def fit_ar1(x):
    xt, xlag = x[1:], x[:-1]
    xbar_, ybar_ = xlag.mean(), xt.mean()
    phi_ = np.sum((xlag - xbar_) * (xt - ybar_)) / np.sum((xlag - xbar_) ** 2)
    c_ = ybar_ - phi_ * xbar_
    return c_, phi_

errors = []
for cutoff in range(4, len(X)):  # need at least 4 points to fit
    train = X[:cutoff]
    actual_next = X[cutoff]
    c_wf, phi_wf = fit_ar1(train)
    forecast = c_wf + phi_wf * train[-1]
    error = actual_next - forecast
    errors.append(error)
    print(f"Train on months 1-{cutoff}: forecast month {cutoff+1} = {forecast:.2f}, "
          f"actual = {actual_next:.2f}, error = {error:.2f}")

rmse = np.sqrt(np.mean(np.array(errors) ** 2))
print(f"\nWalk-forward RMSE: {rmse:.3f}")

Train on months 1-4: forecast month 5 = 6.83, actual = 5.00, error = -1.83
Train on months 1-5: forecast month 6 = 4.00, actual = 6.00, error = 2.00
Train on months 1-6: forecast month 7 = 6.00, actual = 4.00, error = -2.00
Train on months 1-7: forecast month 8 = 4.07, actual = 4.00, error = -0.07

Walk-forward RMSE: 1.686


Each forecast above uses only data that would genuinely have been available at the time, in contrast to a random $k$-fold split, which could easily train on a later month while validating on an earlier one.

## 8b. RMSE and MAPE against a naive benchmark (Section 7.15)

A fitted model's forecasts versus a naive "repeat the last value" benchmark, scored against the same five actual values.

In [13]:
actual_fc = np.array([100, 102, 98, 105, 101])
model_fc = np.array([99, 103, 97, 104, 102])
naive_fc = np.array([98, 100, 102, 98, 105])

def rmse(a, f):
    return np.sqrt(np.mean((a - f) ** 2))

def mape(a, f):
    return 100 * np.mean(np.abs((a - f) / a))

print(f"Model -- RMSE: {rmse(actual_fc, model_fc):.2f}, MAPE: {mape(actual_fc, model_fc):.2f}%")
print(f"Naive -- RMSE: {rmse(actual_fc, naive_fc):.2f}, MAPE: {mape(actual_fc, naive_fc):.2f}%")

Model -- RMSE: 1.00, MAPE: 0.99%
Naive -- RMSE: 4.22, MAPE: 3.73%


## Exercises (match Chapter 7, Suggested Exercises)

3. Add a ninth observation, $X_9=3.0$, refit the AR(1) model, and compare the new coefficients and forecast to the values above.
4. Compute $\sigma_6^2$ if a fifth AssetA return of $r_5=0.035$ were observed.

In [14]:
# Exercise 3: add a ninth observation and refit
X_ex = np.append(X, 3.0)
c_ex, phi_ex = fit_ar1(X_ex)
forecast_ex = c_ex + phi_ex * X_ex[-1]
print(f"Exercise 3 -- phi={phi_ex:.4f}, c={c_ex:.4f}, forecast X10={forecast_ex:.3f}")

# Exercise 4: extend the GARCH recursion one more step
r5 = 0.035
sigma2_6 = omega + alpha_g * r5 ** 2 + beta_g * sigma2[-1]
print(f"Exercise 4 -- sigma_6^2 = {sigma2_6:.6f}, sigma_6 = {np.sqrt(sigma2_6):.2%}")

# Exercise (VAR): shock to Y instead of X
Xv2, Yv2 = 0.0, 1.0
Xv2, Yv2 = A_var @ [Xv2, Yv2]
print(f"Exercise (VAR) -- shock to Y: X1={Xv2:.2f}, Y1={Yv2:.2f}")

# Exercise (RMSE/MAPE): improved naive benchmark
naive2_fc = np.array([99, 101, 99, 104, 100])
print(f"Exercise (naive benchmark) -- RMSE: {rmse(actual_fc, naive2_fc):.2f}, "
      f"MAPE: {mape(actual_fc, naive2_fc):.2f}%")

Exercise 3 -- phi=0.8085, c=0.4468, forecast X10=2.872
Exercise 4 -- sigma_6^2 = 0.000479, sigma_6 = 2.19%
Exercise (VAR) -- shock to Y: X1=0.20, Y1=0.70
Exercise (naive benchmark) -- RMSE: 1.00, MAPE: 0.99%
